# Explosive NSE moves — master research notebook

Rare-event prediction: identify conditions **5–20 sessions** before unusually large upward price moves (MFE-based labels).

**Causal rule:** features at market close **T**; labels from **T+1…T+H** only.

**Order:** Phase 1 audit → panel → events → **pre-event discovery before ML** → features → ablation → walk-forward ML → strategy → scanner.

Artifacts: `results/explosive_moves/`. Override data path with `QUANT_DATA_DIR`.

**Limitations (v1):** no split-adjusted prices; sector/VIX/free-float only when datasets are added.


In [ ]:
from pathlib import Path
import json, warnings, os, sys, gc, subprocess, importlib.util
warnings.filterwarnings("ignore")

from quant.config import DATA_DIR as QUANT_DATA_DIR, PARQUET_DATA_DIR, NIFTY50_PATH

DATA_DIR = PARQUET_DATA_DIR
OUT_DIR = QUANT_DATA_DIR / "results" / "explosive_moves"
for sub in ("panel", "events", "features", "models", "reports", "reports/figures", "scanner"):
    (OUT_DIR / sub).mkdir(parents=True, exist_ok=True)

PANEL_PATH = OUT_DIR / "panel" / "ohlcv_panel.parquet"
EVENTS_PATH = OUT_DIR / "events" / "explosive_events.parquet"
FEATURES_PATH = OUT_DIR / "features" / "feature_dataset.parquet"
AUDIT_PATH = OUT_DIR / "reports" / "data_audit.json"
PRE_EVENT_PATH = OUT_DIR / "reports" / "pre_event_summary.parquet"
ABLATION_PATH = OUT_DIR / "reports" / "feature_ablation.csv"
MODEL_EVAL_PATH = OUT_DIR / "reports" / "model_evaluation.csv"
SCANNER_PATH = OUT_DIR / "scanner" / "latest_ranked.parquet"
SCANNER_CSV = OUT_DIR / "scanner" / "latest_ranked.csv"

assert DATA_DIR.exists(), f"Missing parquet dir: {DATA_DIR}"
print("DATA_DIR:", DATA_DIR)
print("OUT_DIR:", OUT_DIR)
print("NIFTY:", NIFTY50_PATH, "exists" if NIFTY50_PATH.exists() else "missing")


In [ ]:
required = ["pandas", "numpy", "pyarrow", "duckdb", "matplotlib", "scipy"]
missing = [x for x in required if importlib.util.find_spec(x) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + missing)

import pandas as pd
import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq
import duckdb
import matplotlib.pyplot as plt
from scipy import stats

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 180)
pd.options.mode.copy_on_write = True

if importlib.util.find_spec("sklearn") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "scikit-learn"])

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score, brier_score_loss
from sklearn.preprocessing import StandardScaler
from sklearn.calibration import CalibratedClassifierCV

HAS_LGBM = importlib.util.find_spec("lightgbm") is not None
if not HAS_LGBM:
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "lightgbm"])
        HAS_LGBM = True
    except Exception:
        HAS_LGBM = False
if HAS_LGBM:
    import lightgbm as lgb

print("duckdb:", duckdb.__version__, "| lightgbm:", HAS_LGBM)


## Phase 0 — Research configuration


In [ ]:
CFG = {
    "horizons": [5, 10, 20],
    "h_max": 20,
    "event_thresholds": {"E1": (0.10, 5), "E2": (0.20, 10), "E3": (0.30, 20)},
    "extra_thresholds": [0.40, 0.50, 1.00],
    "sustained_retention": {5: 0.05, 10: 0.10, 20: 0.10},
    "triple_barrier": {"up": 0.20, "down": -0.10},
    "min_price": 20.0,
    "min_history_bars": 60,
    "min_median_turnover_20d": 1_000_000.0,
    "eq_series_only": True,
    "top_n_events": 500,
    "pre_event_windows": [5, 10, 20],
    "match_k": 3,
    "random_seed": 42,
    "primary_horizon": 20,
    "primary_mfe_thresh": 0.20,
    "wf_splits": {
        "train_end": 2018,
        "valid_end": 2020,
        "test1_end": 2022,
        "test2_end": 2024,
    },
    "costs": {
        "slippage_bps": 20.0,
        "brokerage_bps": 0.0,
        "stt_bps": 10.0,
        "exchange_bps": 3.25,
        "stamp_bps": 1.5,
        "sebi_bps": 0.0001,
        "gst_rate": 0.18,
    },
    "strategy": {
        "top_k": (10, 20),
        "prob_threshold": 0.25,
        "max_hold": 20,
        "stop_loss": 0.10,
        "max_participation_pct": 0.05,
        "initial_capital": 1_000_000.0,
    },
}

FEATURE_FLAGS = {
    "HAS_DELIVERY": False,
    "HAS_TURNOVER": False,
    "HAS_SERIES": False,
    "HAS_PREV_CLOSE": False,
}

print(json.dumps(CFG, indent=2))


## Phase 1 — Data audit


In [ ]:
def stock_parquet_glob():
    return sorted(
        p for p in DATA_DIR.rglob("nse_cm_*.parquet")
        if "results" not in p.parts and "indices" not in p.parts
    )

files = stock_parquet_glob()
print("Daily stock files:", len(files))
if not files:
    raise FileNotFoundError(f"No nse_cm_*.parquet under {DATA_DIR}")

schema_union = set()
optional_hits = {c: 0 for c in ("turnover", "deliv_qty", "deliv_per", "series", "prev_close")}
for p in [files[0], files[len(files) // 2], files[-1]]:
    names = {c.lower() for c in pq.ParquetFile(p).schema.names}
    schema_union |= names
    df = pd.read_parquet(p)
    for c in optional_hits:
        if c in df.columns:
            optional_hits[c] += 1

con = duckdb.connect()
glob_sql = str(DATA_DIR / "*" / "nse_cm_*.parquet").replace("'", "''")
try:
    audit_sql = (
        f"SELECT COUNT(*) AS n_rows, COUNT(DISTINCT symbol) AS n_symbols, "
        f"MIN(CAST(date AS DATE)) AS dmin, MAX(CAST(date AS DATE)) AS dmax "
        f"FROM read_parquet('{glob_sql}', union_by_name=true)"
    )
    audit_df = con.execute(audit_sql).df()
except Exception as exc:
    subset = [str(p).replace("'", "''") for p in files[: min(300, len(files))]]
    paths_sql = ", ".join(f"'{s}'" for s in subset)
    audit_df = con.execute(
        f"SELECT COUNT(*) AS n_rows FROM read_parquet([{paths_sql}], union_by_name=true)"
    ).df()
    audit_df["note"] = str(exc)

FEATURE_FLAGS["HAS_TURNOVER"] = "turnover" in schema_union
FEATURE_FLAGS["HAS_DELIVERY"] = ("deliv_per" in schema_union) or ("deliv_qty" in schema_union)
FEATURE_FLAGS["HAS_SERIES"] = "series" in schema_union
FEATURE_FLAGS["HAS_PREV_CLOSE"] = "prev_close" in schema_union

if audit_df["dmin"].notna().any():
    dmin, dmax = audit_df.loc[0, "dmin"], audit_df.loc[0, "dmax"]
    print("Date coverage:", dmin, "→", dmax)

audit = {
    "n_files": len(files),
    "schema_union": sorted(schema_union),
    "optional_sample_hits": optional_hits,
    "feature_flags": FEATURE_FLAGS,
    "duckdb_summary": audit_df.to_dict(orient="records"),
    "nifty_exists": NIFTY50_PATH.exists(),
}
AUDIT_PATH.write_text(json.dumps(audit, indent=2, default=str))
display(audit_df)
print("Feature flags:", FEATURE_FLAGS)


## Phase 2 — Panel construction


In [ ]:
def first_series(frame, name):
    s = frame[name]
    return s.iloc[:, 0] if isinstance(s, pd.DataFrame) else s


def normalize_ohlc(df, source_name):
    cols = {str(c).lower().strip(): c for c in df.columns}
    aliases = {
        "date": ["date", "datetime", "timestamp"],
        "symbol": ["symbol", "ticker", "tradingsymbol"],
        "open": ["open"],
        "high": ["high"],
        "low": ["low"],
        "close": ["close", "adj close", "adj_close"],
        "volume": ["volume", "vol", "qty", "quantity", "total traded quantity"],
        "turnover": ["turnover", "tottrdval", "traded_value"],
        "deliv_qty": ["deliv_qty", "deliverablequantity", "delivqty"],
        "deliv_per": ["deliv_per", "delivper", "delivery_percentage"],
        "series": ["series"],
        "prev_close": ["prev_close", "prevclose", "previous_close"],
    }
    out = {}
    for target, names in aliases.items():
        found = next((cols[n] for n in names if n in cols), None)
        if found is not None:
            out[target] = found
    missing = [x for x in ["date", "high", "low", "close"] if x not in out]
    if missing:
        raise ValueError(f"{source_name}: missing {missing}")
    symbol = first_series(df, out["symbol"]) if "symbol" in out else Path(source_name).stem
    x = pd.DataFrame({
        "Date": pd.to_datetime(first_series(df, out["date"]), errors="coerce"),
        "Symbol": pd.Series(symbol, index=df.index).astype(str).str.upper().str.strip(),
        "Open": pd.to_numeric(first_series(df, out["open"]), errors="coerce") if "open" in out else np.nan,
        "High": pd.to_numeric(first_series(df, out["high"]), errors="coerce"),
        "Low": pd.to_numeric(first_series(df, out["low"]), errors="coerce"),
        "Close": pd.to_numeric(first_series(df, out["close"]), errors="coerce"),
        "Volume": pd.to_numeric(first_series(df, out["volume"]), errors="coerce") if "volume" in out else np.nan,
    })
    opt_map = {
        "turnover": "Turnover",
        "deliv_qty": "DelivQty",
        "deliv_per": "DelivPer",
        "prev_close": "PrevClose",
    }
    for src, dst in opt_map.items():
        if src in out:
            x[dst] = pd.to_numeric(first_series(df, out[src]), errors="coerce")
    if "series" in out:
        x["Series"] = first_series(df, out["series"]).astype(str).str.upper().str.strip()
    x = x.dropna(subset=["Date", "Close"]).loc[x["Close"] > 0]
    if getattr(x["Date"].dt, "tz", None) is not None:
        x["Date"] = x["Date"].dt.tz_convert("UTC").dt.tz_localize(None)
    for c in x.select_dtypes(include=["float64"]).columns:
        x[c] = x[c].astype("float32")
    x["Symbol"] = x["Symbol"].astype("string")
    return x


PANEL_EXTRA = ["Turnover", "DelivQty", "DelivPer", "Series", "PrevClose"]
BASE_COLS = ["Date", "Symbol", "Open", "High", "Low", "Close", "Volume"]


def rebuild_panel(force=False):
    if PANEL_PATH.exists() and not force:
        print("Using existing panel:", PANEL_PATH)
        return
    files = stock_parquet_glob()
    writer = None
    n_ok = n_err = 0
    for i, f in enumerate(files, 1):
        try:
            x = normalize_ohlc(pd.read_parquet(f), f.name)
            keep = [c for c in BASE_COLS + PANEL_EXTRA if c in x.columns]
            table = pa.Table.from_pandas(x[keep], preserve_index=False)
            if writer is None:
                writer = pq.ParquetWriter(PANEL_PATH, table.schema, compression="zstd")
            writer.write_table(table)
            n_ok += 1
            if i % 400 == 0:
                gc.collect()
                print(f"stitched {i}/{len(files)}")
        except Exception as exc:
            n_err += 1
            if n_err <= 5:
                print("skip", f.name, exc)
    if writer:
        writer.close()
    print("Panel OK:", n_ok, "errors:", n_err)


rebuild_panel(force=False)
panel = pd.read_parquet(PANEL_PATH)
panel["Date"] = pd.to_datetime(panel["Date"])
panel["Symbol"] = panel["Symbol"].astype("string")
panel = panel.sort_values(["Symbol", "Date"]).reset_index(drop=True)

if "Turnover" not in panel.columns:
    panel["Turnover"] = (panel["Close"] * panel["Volume"]).astype("float32")

if CFG["eq_series_only"] and "Series" in panel.columns:
    panel = panel.loc[panel["Series"].fillna("EQ").eq("EQ")]

nifty = None
if NIFTY50_PATH.exists():
    nifty = normalize_ohlc(pd.read_parquet(NIFTY50_PATH), "NIFTY50")
    nifty = nifty.sort_values("Date")
    nifty["nifty_ret_1"] = nifty["Close"].pct_change()

print(panel.shape, panel["Date"].min(), "→", panel["Date"].max(), "symbols:", panel["Symbol"].nunique())


## Phase 3 — Event labeling (MFE / MAE / forward close return)


In [ ]:
def forward_excursion(g, horizons):
    g = g.sort_values("Date").copy()
    h_max = max(horizons)
    close = g["Close"]
    for H in horizons:
        highs = [g["High"].shift(-k) for k in range(1, H + 1)]
        lows = [g["Low"].shift(-k) for k in range(1, H + 1)]
        g[f"mfe_{H}"] = pd.concat(highs, axis=1).max(axis=1) / close - 1
        g[f"mae_{H}"] = pd.concat(lows, axis=1).min(axis=1) / close - 1
        g[f"ret_{H}"] = close.shift(-H) / close - 1
    tail = g.tail(h_max).index
    for H in horizons:
        g.loc[tail, [f"mfe_{H}", f"mae_{H}", f"ret_{H}"]] = np.nan
    return g


def add_labels(df, horizons, thresholds, cfg):
    df = df.copy()
    for H in horizons:
        thr_list = [thresholds.get(f"E{idx}", (0.2, H))[0] for idx, H2 in [(1, 5), (2, 10), (3, 20)] if H2 == H]
        thr = thresholds.get("E2", (0.2, 10))[0] if H == 10 else {5: 0.10, 20: 0.30}.get(H, 0.20)
        if H == 5:
            thr = 0.10
        elif H == 20:
            thr = 0.30
        mfe = df[f"mfe_{H}"]
        ret = df[f"ret_{H}"]
        df[f"y_explosive_{H}"] = (mfe >= thr).astype("int8")
        sustain = cfg["sustained_retention"].get(H, 0.10)
        df[f"y_sustained_{H}"] = ((mfe >= thr) & (ret >= sustain)).astype("int8")
        up, dn = cfg["triple_barrier"]["up"], cfg["triple_barrier"]["down"]
        df[f"y_tb_{H}"] = ((mfe >= up) & (df[f"mae_{H}"] > dn)).astype("int8")
    for xt in cfg["extra_thresholds"]:
        H = cfg["primary_horizon"]
        df[f"y_mfe{int(xt*100)}_{H}"] = (df[f"mfe_{H}"] >= xt).astype("int8")
    return df


labeled_parts = []
for sym, g in panel.groupby("Symbol", observed=True):
    if len(g) < CFG["min_history_bars"] + CFG["h_max"] + 1:
        continue
    labeled_parts.append(forward_excursion(g, CFG["horizons"]))
labeled = pd.concat(labeled_parts, ignore_index=True)
labeled = add_labels(labeled, CFG["horizons"], CFG["event_thresholds"], CFG)

base_rates = {}
for H in CFG["horizons"]:
    col = f"y_explosive_{H}"
    base_rates[f"H{H}"] = float(labeled[col].mean())
print("Base rates (explosive):", base_rates)

events = labeled.dropna(subset=[f"mfe_{CFG['primary_horizon']}"])
events.to_parquet(EVENTS_PATH, compression="zstd", index=False)
print("Wrote", EVENTS_PATH, "rows", len(events))


## Phase 4 — Pre-event discovery (top gainers vs matched controls)


In [ ]:
rng = np.random.default_rng(CFG["random_seed"])
H = CFG["primary_horizon"]
mfe_col, y_col = f"mfe_{H}", f"y_explosive_{H}"

candidates = events.loc[
    events[mfe_col] >= CFG["primary_mfe_thresh"]
].sort_values(mfe_col, ascending=False).head(CFG["top_n_events"])

if candidates.empty:
    raise ValueError("No explosive events found; loosen thresholds or check panel.")

def liquidity_bucket(turnover):
    if not np.isfinite(turnover) or turnover <= 0:
        return "unk"
    return ["micro", "small", "mid", "large"][min(3, int(np.log10(turnover + 1) // 2))]

candidates = candidates.copy()
candidates["liq_bucket"] = candidates["Turnover"].map(liquidity_bucket)

def match_controls(evt_row, pool, k=3):
    same_day = pool.loc[pool["Date"] == evt_row["Date"]]
    same_day = same_day.loc[same_day[y_col] == 0]
    if same_day.empty:
        return pd.DataFrame()
    lo, hi = evt_row["Close"] * 0.5, evt_row["Close"] * 2.0
    same_day = same_day.loc[(same_day["Close"] >= lo) & (same_day["Close"] <= hi)]
    b = evt_row["liq_bucket"]
    same_day = same_day.loc[same_day["Turnover"].map(liquidity_bucket) == b]
    if len(same_day) <= k:
        return same_day
    return same_day.sample(n=k, random_state=CFG["random_seed"])

pool = events.loc[events["Close"] >= CFG["min_price"]]
ctrl_frames = []
for _, row in candidates.iterrows():
    ctrl = match_controls(row, pool, CFG["match_k"])
    if ctrl.empty:
        continue
    ctrl = ctrl.assign(event_symbol=row["Symbol"], event_date=row["Date"], cohort="control")
    hit = row.to_frame().T.assign(cohort="event")
    ctrl_frames.append(hit)
    ctrl_frames.append(ctrl)

cohort = pd.concat(ctrl_frames, ignore_index=True) if ctrl_frames else candidates.assign(cohort="event")

# Simple pre-event features for comparison
def add_pre_features(df):
    df = df.sort_values(["Symbol", "Date"]).copy()
    g = df.groupby("Symbol", observed=True)
    df["vol_rvol_20"] = df["Volume"] / g["Volume"].transform(lambda s: s.rolling(20, min_periods=10).mean())
    df["px_ret_5"] = g["Close"].pct_change(5)
    df["px_ret_20"] = g["Close"].pct_change(20)
    prev_c = g["Close"].shift(1)
    tr = np.maximum(df["High"] - df["Low"], np.maximum((df["High"] - prev_c).abs(), (df["Low"] - prev_c).abs()))
    df["vola_atr14_pct"] = tr.groupby(df["Symbol"]).transform(lambda s: s.rolling(14, min_periods=5).mean()) / df["Close"] * 100
    if nifty is not None:
        nref = nifty.set_index("Date")["Close"]
        df["nifty_ret_20"] = df["Date"].map(lambda d: nref.pct_change(20).get(d, np.nan))
        df["rs_20"] = df["px_ret_20"] - df["nifty_ret_20"]
    if "DelivPer" in df.columns:
        df["deliv_ma5"] = g["DelivPer"].transform(lambda s: s.rolling(5, min_periods=3).mean())
    return df

feat_cols = ["vol_rvol_20", "px_ret_5", "px_ret_20", "vola_atr14_pct"]
if "rs_20" in cohort.columns:
    feat_cols.append("rs_20")
if "deliv_ma5" in cohort.columns:
    feat_cols.append("deliv_ma5")

cohort = add_pre_features(cohort)
rows = []
for fc in feat_cols:
    if fc not in cohort.columns:
        continue
    a = cohort.loc[cohort["cohort"] == "event", fc].dropna()
    b = cohort.loc[cohort["cohort"] == "control", fc].dropna()
    if len(a) < 30 or len(b) < 30:
        continue
    stat, p = stats.mannwhitneyu(a, b, alternative="two-sided")
    rows.append({"feature": fc, "event_median": float(a.median()), "control_median": float(b.median()), "p_value": float(p)})

pre_summary = pd.DataFrame(rows)
pre_summary.to_parquet(PRE_EVENT_PATH, index=False)
display(pre_summary)

fig, ax = plt.subplots(figsize=(8, 4))
if not pre_summary.empty:
    ax.barh(pre_summary["feature"], pre_summary["event_median"] - pre_summary["control_median"])
    ax.set_title("Event vs control median difference (pre-event)")
    fig.savefig(OUT_DIR / "reports" / "figures" / "pre_event_diff.png", dpi=120, bbox_inches="tight")
plt.show()


## Phase 5 — Feature dataset (stock-day rows)


In [ ]:
def compute_features(panel_df, nifty_df=None):
    x = panel_df.sort_values(["Symbol", "Date"]).copy()
    g = x.groupby("Symbol", observed=True)

    x["vol_rvol_20"] = x["Volume"] / g["Volume"].transform(lambda s: s.rolling(20, min_periods=10).mean())
    x["vol_va"] = g["Volume"].transform(lambda s: s.rolling(5, min_periods=3).mean()) / g["Volume"].transform(lambda s: s.rolling(20, min_periods=10).mean())
    x["vol_turn_accel"] = g["Turnover"].transform(lambda s: s.rolling(5, min_periods=3).mean()) / g["Turnover"].transform(lambda s: s.rolling(20, min_periods=10).mean())

    prev_c = g["Close"].shift(1)
    tr = np.maximum(x["High"] - x["Low"], np.maximum((x["High"] - prev_c).abs(), (x["Low"] - prev_c).abs()))
    x["vola_atr14_pct"] = tr.groupby(x["Symbol"]).transform(lambda s: s.rolling(14, min_periods=5).mean()) / x["Close"] * 100
    x["vola_atr5"] = tr.groupby(x["Symbol"]).transform(lambda s: s.rolling(5, min_periods=3).mean())
    x["vola_atr50"] = tr.groupby(x["Symbol"]).transform(lambda s: s.rolling(50, min_periods=20).mean())
    x["vola_vr"] = x["vola_atr5"] / x["vola_atr50"]
    roll_std = g["Close"].transform(lambda s: s.rolling(20, min_periods=10).std())
    x["vola_bbw"] = (4 * roll_std) / g["Close"].transform(lambda s: s.rolling(20, min_periods=10).mean())

    x["px_ret_5"] = g["Close"].pct_change(5)
    x["px_ret_10"] = g["Close"].pct_change(10)
    x["px_ret_20"] = g["Close"].pct_change(20)
    x["px_dist_20h"] = x["Close"] / g["High"].transform(lambda s: s.rolling(20, min_periods=10).max()) - 1
    x["px_dist_50h"] = x["Close"] / g["High"].transform(lambda s: s.rolling(50, min_periods=20).max()) - 1
    x["px_dist_252h"] = x["Close"] / g["High"].transform(lambda s: s.rolling(252, min_periods=60).max()) - 1
    x["px_clv"] = (x["Close"] - x["Low"]) / (x["High"] - x["Low"]).replace(0, np.nan)
    up = (x["Close"] > g["Close"].shift(1)).astype(int)
    x["px_up_streak"] = up.groupby(x["Symbol"]).transform(lambda s: s.groupby((s != s.shift()).cumsum()).cumsum())

    if "DelivPer" in x.columns:
        x["deliv_ma5"] = g["DelivPer"].transform(lambda s: s.rolling(5, min_periods=3).mean())
        x["deliv_ma20"] = g["DelivPer"].transform(lambda s: s.rolling(20, min_periods=10).mean())
        x["deliv_accel"] = x["deliv_ma5"] / x["deliv_ma20"]
    if "DelivQty" in x.columns:
        x["deliv_qty_accel"] = g["DelivQty"].transform(lambda s: s.rolling(5, min_periods=3).mean()) / g["DelivQty"].transform(lambda s: s.rolling(20, min_periods=10).mean())

    x["liq_turn_med20"] = g["Turnover"].transform(lambda s: s.rolling(20, min_periods=10).median())
    x["liq_zero_ret_20"] = g["Close"].pct_change().transform(lambda s: s.rolling(20, min_periods=10).apply(lambda a: np.mean(np.abs(a) < 1e-8)))
    if "PrevClose" in x.columns:
        x["liq_gap"] = x["Close"] / x["PrevClose"] - 1

    if nifty_df is not None:
        nd = nifty_df[["Date", "Close"]].rename(columns={"Close": "NiftyClose"})
        x = x.merge(nd, on="Date", how="left")
        x["mkt_nifty_ret_5"] = nd.set_index("Date")["NiftyClose"].pct_change(5).reindex(x["Date"]).to_numpy()
        x["mkt_nifty_ret_20"] = nd.set_index("Date")["NiftyClose"].pct_change(20).reindex(x["Date"]).to_numpy()
        x["rs_20"] = x["px_ret_20"] - x["mkt_nifty_ret_20"]
        sma20 = g["Close"].transform(lambda s: s.rolling(20, min_periods=10).mean())
        x_day = x.assign(_above=(x["Close"] > sma20).astype(int))
        breadth = x_day.groupby("Date")["_above"].mean().rename("mkt_breadth_20sma")
        x = x.merge(breadth, on="Date", how="left")

    label_cols = [c for c in events.columns if c.startswith(("mfe_", "mae_", "ret_", "y_"))]
    feat = x.merge(events[["Date", "Symbol"] + label_cols], on=["Date", "Symbol"], how="inner")
    feat = feat.loc[feat["Close"] >= CFG["min_price"]]
    feat = feat.loc[feat["liq_turn_med20"] >= CFG["min_median_turnover_20d"]]
    return feat


features = compute_features(panel, nifty)
feature_cols = [c for c in features.columns if c.startswith(("vol_", "vola_", "px_", "deliv_", "liq_", "rs_", "mkt_"))]
label_cols = [c for c in features.columns if c.startswith("y_")]
print("Features:", len(feature_cols), "Labels:", len(label_cols))
features.to_parquet(FEATURES_PATH, compression="zstd", index=False)
print("Wrote", FEATURES_PATH)


## Phase 6 — Controlled experiments (feature-family ablation)


In [ ]:
FAMILIES = {
    "E1_volume": [c for c in feature_cols if c.startswith("vol_")],
    "E2_volatility": [c for c in feature_cols if c.startswith("vola_")],
    "E3_delivery": [c for c in feature_cols if c.startswith("deliv_")],
    "E4_price_structure": [c for c in feature_cols if c.startswith("px_")],
    "E5_relative_strength": [c for c in feature_cols if c.startswith("rs_")],
    "E6_liquidity": [c for c in feature_cols if c.startswith("liq_")],
    "E7_market": [c for c in feature_cols if c.startswith("mkt_")],
}

TARGET = f"y_explosive_{CFG['primary_horizon']}"


def precision_at_k(y_true, y_score, k=20):
    if len(y_true) == 0:
        return np.nan
    k = min(k, len(y_true))
    idx = np.argsort(-y_score)[:k]
    return float(y_true.iloc[idx].mean())


def lift_at_k(y_true, y_score, k=20):
    base = y_true.mean()
    if base <= 0:
        return np.nan
    return precision_at_k(y_true, y_score, k) / base


def fit_logit_scorer(train_df, cols, target):
    cols = [c for c in cols if c in train_df.columns]
    if not cols:
        return None, cols
    tr = train_df.dropna(subset=cols + [target])
    if tr[target].sum() < 5:
        return None, cols
    X = tr[cols].replace([np.inf, -np.inf], np.nan).fillna(0)
    y = tr[target]
    scaler = StandardScaler()
    Xs = scaler.fit_transform(X)
    w = len(y) / max(y.sum(), 1)
    clf = LogisticRegression(max_iter=500, class_weight={0: 1, 1: w})
    clf.fit(Xs, y)
    return (scaler, clf), cols


def score_df(df, model, cols, target):
    if model is None:
        return pd.Series(np.zeros(len(df)), index=df.index)
    scaler, clf = model
    X = df[cols].replace([np.inf, -np.inf], np.nan).fillna(0)
    return pd.Series(clf.predict_proba(scaler.transform(X))[:, 1], index=df.index)


valid = features.loc[features["Date"].dt.year <= CFG["wf_splits"]["valid_end"]]
valid = valid.loc[valid["Date"].dt.year > CFG["wf_splits"]["train_end"]]
all_cols = feature_cols

full_model, full_cols = fit_logit_scorer(valid, all_cols, TARGET)
scores = score_df(valid, full_model, full_cols, TARGET)
y = valid[TARGET]
full_p20 = precision_at_k(y, scores, 20)
full_lift = lift_at_k(y, scores, 20)

rows = []
for fam, cols in FAMILIES.items():
    ablate_cols = [c for c in all_cols if c not in cols]
    m, used = fit_logit_scorer(valid, ablate_cols, TARGET)
    sc = score_df(valid, m, used, TARGET)
    rows.append({
        "experiment": fam,
        "precision_at_20": precision_at_k(y, sc, 20),
        "lift_at_20": lift_at_k(y, sc, 20),
        "delta_lift_vs_full": lift_at_k(y, sc, 20) - full_lift,
        "n_features_removed": len(cols),
    })
rows.append({"experiment": "full", "precision_at_20": full_p20, "lift_at_20": full_lift, "delta_lift_vs_full": 0, "n_features_removed": 0})
ablation = pd.DataFrame(rows)
ablation.to_csv(ABLATION_PATH, index=False)
display(ablation)


## Phase 7 — ML + purged walk-forward


In [ ]:
def split_masks(df):
    y = df["Date"].dt.year
    s = CFG["wf_splits"]
    train = y <= s["train_end"]
    valid = (y > s["train_end"]) & (y <= s["valid_end"])
    test1 = (y > s["valid_end"]) & (y <= s["test1_end"])
    test2 = (y > s["test1_end"]) & (y <= s["test2_end"])
    holdout = y > s["test2_end"]
    return train, valid, test1, test2, holdout


def purge_overlaps(train_idx, test_start, h_max):
    # Drop training rows whose forward label window could overlap test period
    cutoff = pd.Timestamp(test_start) - pd.Timedelta(days=h_max * 2)
    dates = features.loc[train_idx, "Date"]
    return train_idx & (dates <= cutoff)


train_m, valid_m, test1_m, test2_m, holdout_m = split_masks(features)

Xcols = [c for c in feature_cols if c in features.columns]
TARGET = f"y_explosive_{CFG['primary_horizon']}"


def prep_xy(frame):
    f = frame.dropna(subset=Xcols + [TARGET])
    X = f[Xcols].replace([np.inf, -np.inf], np.nan).fillna(0)
    y = f[TARGET]
    return X, y, f


def eval_probs(y, p, tag):
    return {
        "split": tag,
        "pr_auc": float(average_precision_score(y, p)) if y.sum() > 0 else np.nan,
        "brier": float(brier_score_loss(y, p)),
        "precision_at_20": precision_at_k(y.reset_index(drop=True), pd.Series(p).reset_index(drop=True), 20),
        "lift_at_20": lift_at_k(y.reset_index(drop=True), pd.Series(p).reset_index(drop=True), 20),
        "base_rate": float(y.mean()),
        "n": int(len(y)),
    }

results = []
models = {}

for name, mask in [("valid", valid_m), ("test1", test1_m), ("test2", test2_m), ("holdout", holdout_m)]:
    tr_mask = train_m
    if name != "valid":
        tr_mask = train_m | valid_m
    test_start = features.loc[mask, "Date"].min()
    if pd.notna(test_start):
        tr_mask = purge_overlaps(tr_mask, test_start, CFG["h_max"])
    Xtr, ytr, _ = prep_xy(features.loc[tr_mask])
    Xte, yte, _ = prep_xy(features.loc[mask])
    if len(Xtr) < 100 or len(Xte) < 50:
        continue
    scaler = StandardScaler()
    Xtr_s = scaler.fit_transform(Xtr)
    Xte_s = scaler.transform(Xte)

    logit = LogisticRegression(max_iter=800, class_weight="balanced")
    logit.fit(Xtr_s, ytr)
    cal = CalibratedClassifierCV(logit, cv=3, method="isotonic")
    cal.fit(Xtr_s, ytr)
    p_logit = cal.predict_proba(Xte_s)[:, 1]
    results.append(eval_probs(yte, p_logit, f"logit_{name}"))

    rf = RandomForestClassifier(n_estimators=200, max_depth=8, min_samples_leaf=50, class_weight="balanced", random_state=CFG["random_seed"], n_jobs=-1)
    rf.fit(Xtr, ytr)
    p_rf = rf.predict_proba(Xte)[:, 1]
    results.append(eval_probs(yte, p_rf, f"rf_{name}"))

    if HAS_LGBM:
        scale = (ytr == 0).sum() / max((ytr == 1).sum(), 1)
        lgbm = lgb.LGBMClassifier(
            n_estimators=300,
            learning_rate=0.05,
            num_leaves=63,
            subsample=0.8,
            colsample_bytree=0.8,
            class_weight={0: 1.0, 1: scale},
            random_state=CFG["random_seed"],
        )
        lgbm.fit(Xtr, ytr)
        p_lgb = lgbm.predict_proba(Xte)[:, 1]
        results.append(eval_probs(yte, p_lgb, f"lgbm_{name}"))
        if name == "test2":
            models["lgbm"] = lgbm

model_eval = pd.DataFrame(results)
model_eval.to_csv(MODEL_EVAL_PATH, index=False)
display(model_eval)
if models.get("lgbm") is not None:
    models["lgbm"].booster_.save_model(str(OUT_DIR / "models" / "lgbm_primary.txt"))


## Phase 8 — Executable strategy simulation (research)


In [ ]:
def transaction_cost(notional, side, cfg):
    c = cfg["costs"]
    brokerage = notional * c["brokerage_bps"] / 10000
    stt = notional * c["stt_bps"] / 10000 if side == "SELL" else 0
    exchange = notional * c["exchange_bps"] / 10000
    stamp = notional * c["stamp_bps"] / 10000 if side == "BUY" else 0
    sebi = notional * c["sebi_bps"] / 10000
    gst = (brokerage + exchange + sebi) * c["gst_rate"]
    return brokerage + stt + exchange + stamp + sebi + gst


def execution_price(px, side, cfg):
    slip = cfg["costs"]["slippage_bps"] / 10000
    return px * (1 + slip) if side == "BUY" else px * (1 - slip)


def simulate_topk(features_df, scores, top_k, cfg):
    st = cfg["strategy"]
    f = features_df.copy()
    f["score"] = scores
    dates = sorted(f["Date"].unique())
    cash = st["initial_capital"]
    positions = {}
    equity = []

    for i, d in enumerate(dates[:-1]):
        nxt = dates[i + 1]
        # exits
        for sym in list(positions):
            pos = positions[sym]
            row = panel.loc[(panel["Date"] == nxt) & (panel["Symbol"] == sym)]
            if row.empty:
                continue
            opn, lo = float(row["Open"].iloc[0]), float(row["Low"].iloc[0])
            held = (pd.Timestamp(nxt) - pos["entry_date"]).days
            stop = pos["entry_px"] * (1 - st["stop_loss"])
            sell = False
            if np.isfinite(lo) and lo <= stop:
                sell = True
                raw = min(opn, stop)
            elif held >= st["max_hold"]:
                sell = True
                raw = opn
            if sell and np.isfinite(raw) and raw > 0:
                if pos.get("prev_close") and raw >= pos["prev_close"] * 1.19:
                    continue  # skip likely upper circuit entry impossible
                px = execution_price(raw, "SELL", cfg)
                cash += pos["shares"] * px - transaction_cost(pos["shares"] * px, "SELL", cfg)
                del positions[sym]

        day = f.loc[f["Date"] == d].nlargest(top_k, "score")
        slots = top_k - len(positions)
        for _, r in day.iterrows():
            if slots <= 0:
                break
            sym = r["Symbol"]
            if sym in positions:
                continue
            nxt_row = panel.loc[(panel["Date"] == nxt) & (panel["Symbol"] == sym)]
            if nxt_row.empty:
                continue
            opn = float(nxt_row["Open"].iloc[0])
            prev_c = float(r["Close"])
            if not np.isfinite(opn) or opn <= 0:
                continue
            if prev_c > 0 and opn >= prev_c * 1.195:
                continue
            vol = float(nxt_row["Volume"].iloc[0]) if np.isfinite(nxt_row["Volume"].iloc[0]) else 0
            max_notional = vol * opn * cfg["strategy"]["max_participation_pct"]
            alloc = cash / max(slots, 1)
            notional = min(alloc, max_notional) if max_notional > 0 else alloc
            px = execution_price(opn, "BUY", cfg)
            shares = notional / px
            cost = transaction_cost(notional, "BUY", cfg)
            if cash < notional + cost:
                continue
            cash -= notional + cost
            positions[sym] = {"entry_px": px, "shares": shares, "entry_date": pd.Timestamp(nxt), "prev_close": prev_c}
            slots -= 1

        mtm = cash
        for sym, pos in positions.items():
            row = panel.loc[(panel["Date"] == nxt) & (panel["Symbol"] == sym)]
            if not row.empty:
                mtm += pos["shares"] * float(row["Close"].iloc[0])
        equity.append({"Date": nxt, "equity": mtm, "n_pos": len(positions)})

    return pd.DataFrame(equity)


if "lgbm" in models:
    test = features.loc[test2_m | holdout_m].copy()
    Xte = test[Xcols].replace([np.inf, -np.inf], np.nan).fillna(0)
    test_scores = models["lgbm"].predict_proba(Xte)[:, 1]
    for k in CFG["strategy"]["top_k"]:
        eq = simulate_topk(test, test_scores, k, CFG)
        if not eq.empty:
            ret = eq["equity"].iloc[-1] / eq["equity"].iloc[0] - 1
            print(f"Top-{k} research sim total return: {ret:.2%} (not audited for all biases)")
            eq.to_parquet(OUT_DIR / "reports" / f"equity_top{k}.parquet", index=False)
else:
    print("LightGBM model unavailable; skip strategy simulation.")


## Phase 9 — Daily scanner export


In [ ]:
latest_date = features["Date"].max()
scan = features.loc[features["Date"] == latest_date].copy()

if "lgbm" in models:
    X = scan[Xcols].replace([np.inf, -np.inf], np.nan).fillna(0)
    scan["prob_explosive"] = models["lgbm"].predict_proba(X)[:, 1]
elif full_model is not None:
    scan["prob_explosive"] = score_df(scan, full_model, full_cols, TARGET)
else:
    scan["prob_explosive"] = np.nan

rank_cols = ["Date", "Symbol", "Close", "prob_explosive", "vol_rvol_20", "vola_vr", "px_dist_252h", "rs_20"]
rank_cols = [c for c in rank_cols if c in scan.columns]
scan = scan.sort_values("prob_explosive", ascending=False)
scan[rank_cols].head(50).to_csv(SCANNER_CSV, index=False)
scan.to_parquet(SCANNER_PATH, compression="zstd", index=False)
print("Scanner date", latest_date, "→", SCANNER_PATH)
display(scan[rank_cols].head(20))
